# 4 Tracking, and what the tracking is for

By the end of `02_pretrained.ipynb` you had run four models across two notebooks, and every one of
those numbers now lives in your scrollback. That does not survive lunch, let alone a report.

This notebook does two things at once:

1. introduces **MLflow**, the experiment tracker used for the rest of the course;
2. uses it to answer the question `02_pretrained.ipynb` left open — **at what search budget does
   embedding once pay for itself?**

Those are not separate topics. A tracker only earns its keep once you are running enough
experiments that you cannot hold them in your head, and the whole point of training heads on
cached vectors is that you can suddenly afford to run that many.

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import torch
import torch.optim as optim
from datasets import load_dataset
from torch.utils.data import DataLoader

warnings.simplefilter("ignore", UserWarning)

ARTEFACTS = Path("../../artefacts").resolve()
ARTEFACTS.mkdir(exist_ok=True)

## 4.1 Setting up the store

MLflow needs somewhere to put runs. We use one SQLite file at the root of the repository, so that
runs from every lesson land in the same place and are comparable — and so there is exactly one
file to delete when you want a clean slate.

An **experiment** groups runs that answer the same question. Everything below goes into one, on
purpose: the comparison only means something if the runs sit next to each other on one screen.

In [ ]:
import mlflow

MLRUNS = Path("../../mlruns.db").resolve()
mlflow.set_tracking_uri(f"sqlite:///{MLRUNS}")
mlflow.set_experiment("l2_representations")
print(f"tracking to {MLRUNS}")

## 4.2 The data, and the cache from last notebook

**Flowers**, not fashion. That choice is worth a sentence, because it is the difference between
this notebook having a point and not.

On fashion the two approaches cost almost the same per run — 28x28 greyscale is so small that
training the CNN is already cheap, and measured side by side they came out at about three seconds
each. Nothing to trade off. On flowers the CNN is more than twenty times slower per run than the
head, and *that* is where a search budget starts to mean something. If you want to see the
economics, you have to look where the economics exist.

The cache is the artefact `02_pretrained.ipynb` wrote into `artefacts/`. If you skipped that
notebook the cell below rebuilds it, which is itself the "embed once, reuse many" story: the
expensive step already happened, and this notebook picks the result up off disk.

In [ ]:
from vectormesh import ImageVectorizer, VectorCache
from vectormesh.data import Collate, DatasetSchema, OneHot

N_TRAIN, N_TEST = 4000, 1000
MODEL_NAME = "google/mobilenet_v2_1.0_224"
SIZE = (64, 64)  # what the from-scratch CNN sees; the encoder does its own resizing

flowers = load_dataset("nkirschi/oxford-flowers")
schema = DatasetSchema.infer(flowers["train"])
labels = flowers["train"].features[schema.label_col].names

# shuffle before select -- the train split is sorted by label (see 02_pretrained §3.2)
train = flowers["train"].shuffle(seed=42).select(range(N_TRAIN))
test = flowers["test"].shuffle(seed=42).select(range(min(N_TEST, len(flowers["test"]))))
print(f"{len(labels)} classes | {len(train)} train | {len(test)} test")

In [ ]:
def get_cache(tag: str, dataset) -> tuple[VectorCache, float]:
    """Reuse the cache 02_pretrained built; rebuild only if it is not there."""
    local = sorted(ARTEFACTS.glob(f"*_{tag}_embed"))
    if local:
        print(f"reusing {local[-1].name}")
        return VectorCache.load(local[-1]), 0.0

    print(f"no cache for {tag} yet, building it (this is the expensive step)")
    vectorizer = ImageVectorizer(model_name=MODEL_NAME)
    start = time.time()
    cache = VectorCache.create(
        cache_dir=ARTEFACTS,
        vectorizer=vectorizer,
        dataset=dataset,
        dataset_tag=tag,
        remove_columns=[schema.input_col],
    )
    return cache, time.time() - start


traincache, t1 = get_cache("flowers_train", train)
testcache, t2 = get_cache("flowers_test", test)
ENCODING_SECONDS = t1 + t2
hidden_size = traincache.metadata["embed"]["hidden_size"]
print(f"D = {hidden_size}, encoding cost this session: {ENCODING_SECONDS:.0f}s")

`ENCODING_SECONDS` will read `0` if the cache was already on disk — which is the honest number for
*this* session, and precisely the point. The cost was paid once, by the previous notebook. When we
log it as a run parameter below, remember that a `0` means "somebody else already paid".

In [ ]:
onehot = OneHot(num_classes=len(labels), label_col=schema.label_col, target_col="onehot")
collate_fn = Collate(embedding_col="embed", target_col="onehot", padder=torch.stack)

cached_train = DataLoader(traincache.map(onehot), batch_size=64, shuffle=True, collate_fn=collate_fn)
cached_test = DataLoader(testcache.map(onehot), batch_size=64, shuffle=False, collate_fn=collate_fn)


def pixel_collate(items):
    X = [
        torch.from_numpy(
            np.asarray(i[schema.input_col].convert("RGB").resize(SIZE), dtype=np.float32) / 255.0
        ).permute(2, 0, 1)
        for i in items
    ]
    y = torch.nn.functional.one_hot(
        torch.tensor([i[schema.label_col] for i in items]), len(labels)
    ).float()
    return torch.stack(X), y


pixel_train = DataLoader(train, batch_size=64, shuffle=True, collate_fn=pixel_collate)
pixel_test = DataLoader(test, batch_size=64, shuffle=False, collate_fn=pixel_collate)

print("cached batch:", tuple(next(iter(cached_train))[0].shape))
print("pixel batch: ", tuple(next(iter(pixel_train))[0].shape))

## 4.3 One training function, two models

`mltrainer` already knows how to talk to MLflow: `ReportTypes.MLFLOW` makes the `Trainer` log the
loss, every metric and the learning rate on every epoch. What it does *not* know is anything about
your experiment design — the hyperparameters, the tags, the cost of building the cache. Those you
log yourself, inside a `with mlflow.start_run()` block.

Note that we set exactly **one** report type. It is tempting to switch on TensorBoard "as well,
just in case", and then you have two half-populated stores and no idea which one is authoritative.
Pick one per notebook.

In [ ]:
from cnn import CNN  # same file 02_pretrained imported -- one model, two notebooks
from mltrainer import ReportTypes, Trainer, TrainerSettings

from vectormesh.components import GaussianNoise, NeuralNet, Serial
from vectormesh.components.metrics import Accuracy
from vectormesh.data.vectorizers import detect_device

device = detect_device()
EPOCHS = 5
modeldir = Path("modellog").resolve()
modeldir.mkdir(exist_ok=True)


def count_params(model) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def run(model, trainloader, testloader, *, kind: str, params: dict, encoding_seconds: float):
    """Train one model and record it as one MLflow run. Returns (accuracy, seconds)."""
    with mlflow.start_run():
        mlflow.set_tag("kind", kind)
        mlflow.log_params(params)
        mlflow.log_param("trainable_parameters", count_params(model))
        mlflow.log_param("encoding_seconds", round(encoding_seconds))

        settings = TrainerSettings(
            epochs=EPOCHS,
            metrics=[Accuracy()],
            logdir=modeldir,
            train_steps=len(trainloader),
            valid_steps=len(testloader),
            reporttypes=[ReportTypes.MLFLOW],
            earlystop_kwargs=None,
        )
        trainer = Trainer(
            model=model,
            settings=settings,
            loss_fn=torch.nn.CrossEntropyLoss(),
            optimizer=optim.Adam,
            traindataloader=trainloader,
            validdataloader=testloader,
            scheduler=optim.lr_scheduler.ReduceLROnPlateau,
            device=device,
        )
        start = time.time()
        trainer.loop()
        elapsed = time.time() - start

        metric_dict, _ = trainer.evalbatches()
        accuracy = metric_dict["Accuracy"]
        mlflow.log_metric("final_accuracy", accuracy)
        mlflow.log_metric("train_seconds", elapsed)
        return accuracy, elapsed

### The two baseline runs

Same data, same trainer, same epochs. One reads pixels and learns its own filters; one reads
cached vectors and learns a head on top of somebody else's.

In [ ]:
cnn_params = dict(filters=128, units1=128, units2=128)
cnn = CNN(input_size=(64, 3, *SIZE), num_classes=len(labels), **cnn_params)
cnn_acc, cnn_time = run(
    cnn, pixel_train, pixel_test,
    kind="cnn_from_scratch", params=cnn_params, encoding_seconds=0,
)

head_params = dict(sigma=0.1)
head = Serial([
    GaussianNoise(sigma=head_params["sigma"]),
    NeuralNet(hidden_size=hidden_size, out_size=len(labels)),
])
head_acc, head_time = run(
    head, cached_train, cached_test,
    kind="frozen_head", params=head_params, encoding_seconds=ENCODING_SECONDS,
)

print(f"\nCNN  : accuracy {cnn_acc:.3f} in {cnn_time:.0f}s")
print(f"head : accuracy {head_acc:.3f} in {head_time:.0f}s")

Two runs, one experiment, and the difference between them is now a row in a database rather than a
line in your terminal. Go and look at it (§4.5) before reading on.

## 4.4 The search, and the arithmetic underneath it

`hyperopt` searches a space of hyperparameters with **TPE** (Tree-structured Parzen Estimator): it
models the objective as a distribution, learns from the trials it has already run, and spends its
remaining budget where the model thinks the good region is. That beats a grid, and it beats
guessing.

But look at what a search actually *does* to the two approaches:

- searching the CNN retrains the convolutions on every trial → total cost `n * cnn_time`
- searching the head reuses the cache on every trial → total cost `encoding + n * head_time`

The first term of the second expression is paid once. The second term is multiplied. **A search
budget is a multiplier on the per-trial cost, and the whole reason to cache is to make the thing
being multiplied small.** That is the argument for the entire structure of this course, and it is
one line of arithmetic.

In [ ]:
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from hyperopt.pyll import scope


def objective_head(params):
    model = Serial([
        GaussianNoise(sigma=params["sigma"]),
        NeuralNet(hidden_size=hidden_size, out_size=len(labels)),
    ])
    accuracy, _ = run(
        model, cached_train, cached_test,
        kind="search_head", params=params, encoding_seconds=ENCODING_SECONDS,
    )
    return {"loss": -accuracy, "status": STATUS_OK}


head_space = {"sigma": hp.uniform("sigma", 0.0, 0.5)}

start = time.time()
best_head = fmin(fn=objective_head, space=head_space, algo=tpe.suggest,
                 max_evals=5, trials=Trials())
head_search_time = time.time() - start
best_head, f"{head_search_time:.0f}s"

Five trials, and each one is a fully tracked run. Notice how little that cost.

We are deliberately *not* running the equivalent search over the CNN here — that is exercise E1,
and it is more instructive to predict the number first and then check it. You already have
everything you need to predict it, from the two baseline runs above.

**One trap to know about before you write your own search space**, because it will not announce
itself: a search space is part of the model's contract, and `hyperopt` will cheerfully sample a
configuration that does not build. `mltrainer`'s `CNNblocks` (which you will meet again in lesson
4) computes the size of its flatten layer from the pooling alone:

```python
matrix_size = (matrixshape[0] // pool**num_maxpools) * (matrixshape[1] // pool**num_maxpools)
```

That is only correct while every convolution preserves its input size — and `ConvBlock` hardcodes
`padding=1`, so it holds for `kernel_size=3` and nothing else. Put `kernel_size` in the space and
trials die mid-run with a shape error.

The version of this notebook that this one replaces did list `kernel_size` in its search space —
and its objective then hardcoded `kernel_size=3` and ignored the sampled value. So the study
reported three dimensions and explored one, and nothing failed. That is the more dangerous bug:
a search that looks like it is doing something.

### Where is the crossover?

You now have every number you need. The cost of running `n` trials is:

- **CNN**: `n * cnn_time`
- **head**: `encoding + n * head_time`

They cross at `n = encoding / (cnn_time - head_time)`. Below that many trials, training from
scratch is cheaper and caching is not worth the bother. Above it, every extra trial is nearly free.

In [ ]:
# the per-trial costs are the two baseline runs from 4.3 -- same data, same epochs
per_trial_cnn = cnn_time
per_trial_head = head_time

# ENCODING_SECONDS is 0 when 02_pretrained already built the cache, which would make the
# crossover trivially zero and hide the whole trade-off. Substitute what you measured there.
REFERENCE_ENCODING_SECONDS = 60.0
encoding = ENCODING_SECONDS if ENCODING_SECONDS > 0 else REFERENCE_ENCODING_SECONDS
if ENCODING_SECONDS == 0:
    print(f"(cache already existed; using {REFERENCE_ENCODING_SECONDS:.0f}s as the reference)\n")

print(f"per trial, CNN from scratch : {per_trial_cnn:6.1f}s")
print(f"per trial, head on cache    : {per_trial_head:6.1f}s")
print(f"one-time encoding           : {encoding:6.1f}s")

if per_trial_cnn > per_trial_head:
    crossover = encoding / (per_trial_cnn - per_trial_head)
    print(f"\ncrossover at n = {crossover:.1f} trials\n")
    print(f"  {'trials':>7} {'CNN':>10} {'head':>10}")
    for n in (1, 5, 20, 100):
        print(f"  {n:>7} {n * per_trial_cnn:>9.0f}s {encoding + n * per_trial_head:>9.0f}s")
else:
    print("\nno crossover: the head is not cheaper per trial here -- explain why!")

Read that table as a *shape*, not as a set of numbers — the numbers depend on your laptop, on how
much you subsampled, and on how big the encoder is.

The shape is: one column grows steeply from zero, the other starts at the encoding cost and grows
slowly. The interesting consequence is that **the more experiments you intend to run, the more
obviously you should cache** — and "run more experiments" is not a nice-to-have, it is how you
find out anything at all. A method that makes each experiment ten times cheaper does not just save
you time; it changes which questions you are willing to ask.

That is what lesson 4 is about, and why it comes after this one.

## 4.5 The dashboard

MLflow's real value is the UI. Start the server, pointed at the same store the notebook wrote to:

```bash
mlflow server --backend-store-uri sqlite:///../../mlruns.db --host 127.0.0.1 --port 5000
```

Then open `http://127.0.0.1:5000`. There is a `Makefile` in this folder with the same command
(`make show_logs`), and a `stop_server.sh` for when port 5000 is stuck. On Windows the `\`
line continuations will not work — put the command on one line.

Things worth doing there, in order:

1. filter on `tags.kind` to see the four kinds of run side by side;
2. tick two runs and hit **Compare** — parameters that differ are highlighted for you;
3. plot `final_accuracy` against `train_seconds`, and then against `encoding_seconds`.

![](mlflow_overview.png)

![](mlflow_viz.png)

## Exercises

**E1 — run the other search, but predict it first.** Write down, before running anything, how long
a 5-trial `hyperopt` search over the *CNN* will take, using `cnn_time` from §4.3. Then write the
search: an `objective_cnn` mirroring `objective_head`, over `filters`, `units1` and `units2` (all
three are safe — every point in that space builds, which is not true of every model, see §4.4).
Run it with `max_evals=5` and compare against your prediction.

Two follow-ups. Was your estimate high or low, and why? And now that both searches are in the same
MLflow experiment, which of the two found the better model — and is that the same question as
which was cheaper?

**E2 — search something that matters.** The head search only varied `sigma`. Extend it to the head
architecture — `NeuralNet` takes a `hidden_size`, and `Serial` will happily hold more components.
Does anything beat "one `GaussianNoise` and one `NeuralNet`"? Log every trial, and answer from the
dashboard rather than from your memory of the last run.

**E3 — the case against caching.** Construct a situation where the crossover is so high that
caching is the wrong call, and check your reasoning against the arithmetic in §4.4. Two starting
points: a very cheap model to train, or a very expensive encoder. Which of the two is the more
realistic reason to skip it?

**E4 — what the tracker cannot tell you.** Look back at the four `kind`s of run in the dashboard.
Nothing recorded there says *which encoder produced the cache*. Where does that information live,
and what would you have to log for a run to be reproducible six months from now?